In [1]:
#command to mount my drive on colab

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!apt-get update -qq
!apt-get install -y openjdk-21-jdk-headless -qq

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../openjdk-21-jdk-headless_21.0.13.1+2-1~24.04.1_amd64.deb ...
Unpacking openjdk-21-jdk-headless:amd64 (21.0.13.1+2-1~24.04.1) over (21.0.12.1+1-1~24.04.4) ...
Preparing to unpack .../openjdk-21-jre-headless_21.0.13.1+2-1~24.04.1_amd64.deb ...
Unpacking openjdk-21-jre-headless:amd64 (21.0.13.1+2-1~24.04.1) over (21.0.12.1+1-1~24.04.4) ...
Setting up openjdk-21-jre-headless:amd64 (21.0.13.1+2-1~24.04.1) ...
Installing new version of config file /etc/java-21-openjdk/security/blocked.certs ...
Installing new version of config file /etc/java-21-openjdk/security/default.policy ...
Installing new version of config file /etc/java-21-openjdk/security/java.security ...
Processing triggers for ca-certificates-java (20

In [3]:
#Setting/activating the environment variable as java (not compulsory everytime if you are on same runtime)
import os
os.environ["JAVA_HOME"]="/lib/jvm/java-21-openjdk-amd64"

In [4]:
from pyspark import SparkConf, SparkContext
from pyspark.sql import SQLContext
conf = SparkConf().set('spark.ui.port', '4050').setAppName("Housing").setMaster("local[2]")
sc = SparkContext.getOrCreate(conf=conf)
sqlContext = SQLContext(sc)

/usr/local/lib/python3.13/dist-packages/pyspark/sql/context.py:112: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


In [5]:
house_df = sqlContext.read.format('com.databricks.spark.csv').options(header='true', inferschema='true').load(r'Housing.csv')
house_df.printSchema()

root
 |-- price: integer (nullable = true)
 |-- area: integer (nullable = true)
 |-- bedrooms: integer (nullable = true)
 |-- bathrooms: integer (nullable = true)
 |-- stories: integer (nullable = true)
 |-- mainroad: string (nullable = true)
 |-- guestroom: string (nullable = true)
 |-- basement: string (nullable = true)
 |-- hotwaterheating: string (nullable = true)
 |-- airconditioning: string (nullable = true)
 |-- parking: integer (nullable = true)
 |-- prefarea: string (nullable = true)
 |-- furnishingstatus: string (nullable = true)



In [6]:
house_df.toPandas()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished
...,...,...,...,...,...,...,...,...,...,...,...,...,...
540,1820000,3000,2,1,1,yes,no,yes,no,no,2,no,unfurnished
541,1767150,2400,3,1,1,no,no,no,no,no,0,no,semi-furnished
542,1750000,3620,2,1,1,yes,no,no,no,no,0,no,unfurnished
543,1750000,2910,3,1,1,no,no,no,no,no,0,no,furnished


In [7]:
house_df.dtypes

[('price', 'int'),
 ('area', 'int'),
 ('bedrooms', 'int'),
 ('bathrooms', 'int'),
 ('stories', 'int'),
 ('mainroad', 'string'),
 ('guestroom', 'string'),
 ('basement', 'string'),
 ('hotwaterheating', 'string'),
 ('airconditioning', 'string'),
 ('parking', 'int'),
 ('prefarea', 'string'),
 ('furnishingstatus', 'string')]

In [10]:
for i in house_df.dtypes:
  if i[1]=="string":
    print(i[0],end=" ")

mainroad guestroom basement hotwaterheating airconditioning prefarea furnishingstatus 

In [11]:
# prompt: select most columns except 2 as features using vector assembler

from pyspark.ml.feature import VectorAssembler

# Excluding '_c0' (index col) and 'medv' (target) from features
assembler = VectorAssembler(inputCols=[col for col in house_df.columns if col not in ['mainroad','guestroom','basement','hotwaterheating','airconditioning','prefarea','furnishingstatus','price']],outputCol='features')

assembled_df = assembler.transform(house_df)
assembled_df.select('features').show(truncate=False)
vhouse_df = assembled_df.select(['features', 'price'])
vhouse_df.show(3)

+-------------------------+
|features                 |
+-------------------------+
|[7420.0,4.0,2.0,3.0,2.0] |
|[8960.0,4.0,4.0,4.0,3.0] |
|[9960.0,3.0,2.0,2.0,2.0] |
|[7500.0,4.0,2.0,2.0,3.0] |
|[7420.0,4.0,1.0,2.0,2.0] |
|[7500.0,3.0,3.0,1.0,2.0] |
|[8580.0,4.0,3.0,4.0,2.0] |
|[16200.0,5.0,3.0,2.0,0.0]|
|[8100.0,4.0,1.0,2.0,2.0] |
|[5750.0,3.0,2.0,4.0,1.0] |
|[13200.0,3.0,1.0,2.0,2.0]|
|[6000.0,4.0,3.0,2.0,2.0] |
|[6550.0,4.0,2.0,2.0,1.0] |
|[3500.0,4.0,2.0,2.0,2.0] |
|[7800.0,3.0,2.0,2.0,0.0] |
|[6000.0,4.0,1.0,2.0,2.0] |
|[6600.0,4.0,2.0,2.0,1.0] |
|[8500.0,3.0,2.0,4.0,2.0] |
|[4600.0,3.0,2.0,2.0,2.0] |
|[6420.0,3.0,2.0,2.0,1.0] |
+-------------------------+
only showing top 20 rows
+--------------------+--------+
|            features|   price|
+--------------------+--------+
|[7420.0,4.0,2.0,3...|13300000|
|[8960.0,4.0,4.0,4...|12250000|
|[9960.0,3.0,2.0,2...|12250000|
+--------------------+--------+
only showing top 3 rows


In [12]:
splits = vhouse_df.randomSplit([0.7, 0.3])
train_df = splits[0]
test_df = splits[1]
#train_df,test_df=vhouse_df.randomSplit([0.7,0.3])

In [15]:
from pyspark.ml.regression import LinearRegression
lr = LinearRegression(featuresCol = 'features', labelCol='price', maxIter=10)
lr_model = lr.fit(train_df)#
lr_predictions = lr_model.transform(test_df)
lr_predictions.select("prediction","price","features").show()
from pyspark.ml.evaluation import RegressionEvaluator
lr_evaluator = RegressionEvaluator(predictionCol="prediction", labelCol="price",metricName="r2")
print("R Squared (R2) on test data = %g" % lr_evaluator.evaluate(lr_predictions))

+------------------+-------+--------------------+
|        prediction|  price|            features|
+------------------+-------+--------------------+
|3131210.5496655083|1890000|[1700.0,3.0,1.0,2...|
| 3465594.633581241|4340000|[1905.0,5.0,1.0,2...|
|3236580.7730795047|3850000|[2015.0,3.0,1.0,2...|
|3280066.8970281375|3290000|[2145.0,3.0,1.0,2...|
| 3928707.637119817|3640000|[2275.0,3.0,1.0,3...|
| 2660757.944361838|3290000|[2500.0,2.0,1.0,1...|
|    5871125.955709|4200000|[2610.0,4.0,3.0,2...|
|  3118530.17083509|3220000|[2684.0,2.0,1.0,1...|
|2889666.8314621877|4235000|[2787.0,3.0,1.0,1...|
| 4779030.169690562|2380000|[2787.0,4.0,2.0,2...|
| 2894015.443857051|2660000|[2800.0,3.0,1.0,1...|
|3525930.7516607945|4403000|[2880.0,3.0,1.0,2...|
| 2797906.489122912|2604000|[2910.0,2.0,1.0,1...|
|3566071.7891518404|4900000|[3000.0,3.0,1.0,2...|
| 3578114.100399154|3500000|[3036.0,3.0,1.0,2...|
| 2841392.613071545|2870000|[3040.0,2.0,1.0,1...|
|3606212.8266428863|3220000|[3120.0,3.0,1.0,2...|


In [16]:
from pyspark.ml.regression import DecisionTreeRegressor
dt = DecisionTreeRegressor(featuresCol ='features', labelCol = 'price')
dt_model = dt.fit(train_df)
dt_predictions = dt_model.transform(test_df)
dt_evaluator = RegressionEvaluator(labelCol="price", predictionCol="prediction", metricName="rmse")
rmse = dt_evaluator.evaluate(dt_predictions)
print("Root Mean Squared Error (RMSE) on test data = %g" % rmse)

Root Mean Squared Error (RMSE) on test data = 1.22797e+06


In [17]:
dt_evaluator = RegressionEvaluator(labelCol="price", predictionCol="prediction", metricName="r2")
r2 = dt_evaluator.evaluate(dt_predictions)
print("R2 on test data = %g" % r2)

R2 on test data = 0.539205
